In [ ]:
from databricks.sdk import WorkspaceClient
from databricks.sdk.service.pipelines import (
    IngestionPipelineDefinition,
    IngestionConfig,
    TableSpec,
)

w = WorkspaceClient()
#data_engineering_workshop.sf_datadata_engineering_workshop.sf_eventlog
SALESFORCE_CONNECTION_NAME = "lakeflow"          # ← from connection_name
DESTINATION_CATALOG        = "data_engineering_workshop"         # ← from destination_catalog 
DESTINATION_SCHEMA         = "sf_data"  # ← from destination_schema 
EVENT_LOG_CATALOG          = "data_engineering_workshop"         # ← same catalog
EVENT_LOG_SCHEMA           = "sf_eventlog"  
PIPELINE_NAME              = "sfdc_account_code" # ← new name for UI pipeline

In [ ]:
pipeline = w.pipelines.create(
    name=PIPELINE_NAME,
    catalog=EVENT_LOG_CATALOG,
    target=EVENT_LOG_SCHEMA,
    ingestion_definition=IngestionPipelineDefinition(
        connection_name=SALESFORCE_CONNECTION_NAME,
        objects=[
            IngestionConfig(
                table=TableSpec(
                    source_schema="objects",
                    source_table="Account",
                    destination_catalog=DESTINATION_CATALOG,
                    destination_schema=DESTINATION_SCHEMA,
                )
            )
        ]
    ),
    continuous=False,
)

pipeline_id = pipeline.pipeline_id
print(f" Pipeline created! ID: {pipeline_id}")

In [ ]:
import time

run = w.pipelines.start_update(pipeline_id=pipeline_id, full_refresh=False)
update_id = run.update_id
print(f"Run triggered! Update ID: {update_id}")

while True:
    status = w.pipelines.get_update(pipeline_id=pipeline_id, update_id=update_id)
    state  = status.update.state.value
    print(f"   State: {state}")
    if state in ("COMPLETED", "FAILED", "CANCELED"):
        break
    time.sleep(15)

print(f"{'Done!' if state == 'COMPLETED' else 'Failed: ' + state}")

In [ ]:
df = spark.table(f"{DESTINATION_CATALOG}.{DESTINATION_SCHEMA}.Account")
print(f"Total Account records: {df.count()}")
display(df.limit(10))